# Prediksi Diabetes Belum Terdiagnosis Berbasis Fitur Non-Laboratorium
## Validasi Temporal Lintas Siklus NHANES 2013–2023

Notebook ini mereproduksi seluruh analisis pada artikel *Prediksi Diabetes Belum Terdiagnosis Berbasis Fitur Non-Laboratorium: Validasi Temporal Lintas Siklus NHANES 2013–2023* (Ade Rumintarsih, Universitas Nusa Mandiri).

Alur notebook:
1. Mengunduh data NHANES (opsional) atau memuat data gabungan yang disertakan di folder `data/`
2. Seleksi partisipan dan definisi luaran
3. Statistik deskriptif, uji antarsiklus, uji bivariat, dan regresi logistik multivariabel
4. Pelatihan model (LR, RF, GB, MLP) dan validasi temporal (2017–2018 dan 2021–2023)
5. Kalibrasi, rekalibrasi, permutation importance, subkelompok, dan analisis sensitivitas berbobot
6. Gambar 1–4

Seluruh hasil disimpan ke folder `outputs/`.

In [ ]:
import os, json, pickle, warnings, urllib.request
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
os.makedirs("outputs", exist_ok=True); os.makedirs("data/raw", exist_ok=True)
DATA_FILE = "data/nhanes_HIJL_adults.csv.gz"
DOWNLOAD = False   # ubah ke True untuk mengunduh ulang data mentah dari CDC

## 1. Data NHANES

Data gabungan `data/nhanes_HIJL_adults.csv.gz` berisi responden berusia ≥20 tahun dari siklus 2013–2014 (H), 2015–2016 (I), 2017–2018 (J), dan 2021–2023 (L). Data tersebut dibuat dari berkas XPT resmi CDC. Untuk membuatnya ulang dari sumber aslinya, ubah `DOWNLOAD = True` pada sel sebelumnya.

In [ ]:
CYCLES = {"H": 2013, "I": 2015, "J": 2017, "L": 2021}
SPEC = {
 "DEMO": ["SEQN","RIDAGEYR","RIAGENDR","RIDRETH3","DMDEDUC2","INDFMPIR","RIDEXPRG","WTMEC2YR","SDMVPSU","SDMVSTRA"],
 "DIQ":  ["SEQN","DIQ010","DIQ050","DIQ070","DIQ160"],
 "GHB":  ["SEQN","LBXGH"],
 "GLU":  ["SEQN","LBXGLU","WTSAF2YR"],
 "BMX":  ["SEQN","BMXBMI","BMXWAIST","BMXHT","BMXWT"],
 "BPX":  ["SEQN","BPXSY1","BPXSY2","BPXSY3","BPXDI1","BPXDI2","BPXDI3"],
 "BPXO": ["SEQN","BPXOSY1","BPXOSY2","BPXOSY3","BPXODI1","BPXODI2","BPXODI3"],
 "MCQ":  ["SEQN","MCQ300C"],
 "BPQ":  ["SEQN","BPQ020","BPQ080"],
 "SMQ":  ["SEQN","SMQ020"],
 "PAQ":  ["SEQN","PAQ605","PAQ620","PAQ650","PAQ665","PAD680"],
}
COLS = ["cycle","SEQN","RIDAGEYR","RIAGENDR","RIDRETH3","DMDEDUC2","INDFMPIR","RIDEXPRG","WTMEC2YR","SDMVPSU","SDMVSTRA",
        "DIQ010","DIQ050","DIQ070","DIQ160","LBXGH","LBXGLU","WTSAF2YR","BMXBMI","BMXWAIST","BMXHT","BMXWT",
        "BPXSY1","BPXSY2","BPXSY3","BPXDI1","BPXDI2","BPXDI3","BPXOSY1","BPXOSY2","BPXOSY3","BPXODI1","BPXODI2","BPXODI3",
        "MCQ300C","BPQ020","BPQ080","SMQ020","PAQ605","PAQ620","PAQ650","PAQ665","PAD680"]

def read_xpt(name, year):
    path = f"data/raw/{name}.xpt"
    if not os.path.exists(path):
        url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt"
        urllib.request.urlretrieve(url, path)
    return pd.read_sas(path, format="xport")

if DOWNLOAD:
    parts = []
    for c, yr in CYCLES.items():
        m = None
        for f, keep in SPEC.items():
            if (f == "BPX" and c == "L") or (f == "BPXO" and c != "L"):
                continue
            df = read_xpt(f"{f}_{c}", yr)
            df = df[[k for k in keep if k in df.columns]]
            m = df if m is None else m.merge(df, on="SEQN", how="left")
        m["cycle"] = c
        parts.append(m[m.RIDAGEYR >= 20])
    raw = pd.concat(parts, ignore_index=True).reindex(columns=COLS)
    raw.to_csv(DATA_FILE, index=False)

d = pd.read_csv(DATA_FILE)
print(d.shape); d.groupby("cycle").size()

## 2. Seleksi partisipan dan definisi luaran

In [ ]:
flow = {"adults_20plus": d.groupby("cycle").size().to_dict()}
d = d[d.RIDEXPRG.ne(1)];                 flow["after_excl_pregnant"] = d.groupby("cycle").size().to_dict()
d = d[d.DIQ010.isin([1, 2, 3])];         flow["after_excl_invalid_DIQ010"] = d.groupby("cycle").size().to_dict()
diag = (d.DIQ010 == 1) | (d.DIQ050 == 1) | (d.DIQ070 == 1)
flow["diagnosed_excluded"] = d[diag].groupby("cycle").size().to_dict()
d = d[~diag]
d = d[d.LBXGH.notna()];                  flow["final_with_HbA1c"] = d.groupby("cycle").size().to_dict()
d["y"] = ((d.LBXGH >= 6.5) | (d.LBXGLU >= 126)).astype(int)   # diabetes belum terdiagnosis
pd.DataFrame(flow)

## 3. Fitur non-laboratorium

In [ ]:
f = pd.DataFrame(index=d.index)
f["age"] = d.RIDAGEYR
f["male"] = (d.RIAGENDR == 1).astype(int)
f["race"] = d.RIDRETH3.map({1:"Mexican American",2:"Other Hispanic",3:"NH White",4:"NH Black",6:"NH Asian",7:"Other/Multi"})
f["education"] = d.DMDEDUC2.where(d.DMDEDUC2.isin([1,2,3,4,5]))
f["pir"] = d.INDFMPIR
f["bmi"] = d.BMXBMI
f["waist"] = d.BMXWAIST
sbp_a = d[["BPXSY1","BPXSY2","BPXSY3"]].replace(0, np.nan).mean(axis=1)
dbp_a = d[["BPXDI1","BPXDI2","BPXDI3"]].replace(0, np.nan).mean(axis=1)
sbp_o = d[["BPXOSY1","BPXOSY2","BPXOSY3"]].mean(axis=1)
dbp_o = d[["BPXODI1","BPXODI2","BPXODI3"]].mean(axis=1)
f["sbp"] = np.where(d.cycle == "L", sbp_o, sbp_a)     # 2021–2023: osilometrik
f["dbp"] = np.where(d.cycle == "L", dbp_o, dbp_a)
yn = lambda s: s.map({1: 1, 2: 0})
f["htn_hx"] = yn(d.BPQ020); f["chol_hx"] = yn(d.BPQ080); f["ever_smoke"] = yn(d.SMQ020)
f["fam_hx"] = yn(d.MCQ300C)                           # tidak ditanyakan pada 2021–2023
f["sedentary_h"] = d.PAD680.where(d.PAD680 <= 1440) / 60
f["cycle"] = d.cycle; f["y"] = d.y; f["w"] = d.WTMEC2YR
f.to_csv("outputs/analytic_dataset.csv", index=False)
NUM_FULL = ["age","male","education","pir","bmi","waist","sbp","dbp","htn_hx","chol_hx","ever_smoke","sedentary_h","fam_hx"]
NUM_RED = [c for c in NUM_FULL if c != "fam_hx"]
CAT = ["race"]
train = f[f.cycle.isin(["H","I"])]; testJ = f[f.cycle == "J"]; testL = f[f.cycle == "L"]
print(len(train), len(testJ), len(testL))

## 4. Statistik deskriptif dan uji antarsiklus (Tabel 1)

In [ ]:
prev = pd.DataFrame([dict(cycle=c, n=len(g), cases=int(g.y.sum()), prev=g.y.mean(), weighted_prev=np.average(g.y, weights=g.w))
                     for c, g in f.groupby("cycle")])
display(prev)
CONT = ["age","bmi","waist","sbp","dbp","pir","sedentary_h"]
rows = []
for v in CONT:
    g = [x[v].dropna() for _, x in f.groupby("cycle")]
    F, p = stats.f_oneway(*g); allv = pd.concat(g)
    eta2 = sum(len(x)*(x.mean()-allv.mean())**2 for x in g) / ((allv-allv.mean())**2).sum()
    rows.append(dict(var=v, test=f"F = {F:.1f}", p=p, effect=f"eta2 = {eta2:.3f}"))
for v in ["male","htn_hx","chol_hx","ever_smoke","y","race"]:
    ct = pd.crosstab(f[v], f.cycle); chi, p, dof, _ = stats.chi2_contingency(ct)
    V = np.sqrt(chi / (ct.values.sum()*(min(ct.shape)-1)))
    rows.append(dict(var=v, test=f"chi2({dof}) = {chi:.1f}", p=p, effect=f"V = {V:.3f}"))
cyc = pd.DataFrame(rows); cyc.to_csv("outputs/cycle_differences.csv", index=False); cyc

## 5. Uji bivariat kasus vs bukan kasus (data latih)

In [ ]:
out = []
for v in CONT:
    a = train.loc[train.y==1, v].dropna(); b = train.loc[train.y==0, v].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)     # uji t Welch
    sp = np.sqrt(((len(a)-1)*a.var()+(len(b)-1)*b.var())/(len(a)+len(b)-2))
    dfw = (a.var()/len(a)+b.var()/len(b))**2/((a.var()/len(a))**2/(len(a)-1)+(b.var()/len(b))**2/(len(b)-1))
    out.append(dict(var=v, case=f"{a.mean():.1f} ({a.std():.1f})", noncase=f"{b.mean():.1f} ({b.std():.1f})",
                    test=f"t({dfw:.0f}) = {t:.2f}", p=p, effect=f"d = {(a.mean()-b.mean())/sp:.2f}"))
for v in ["male","htn_hx","chol_hx","ever_smoke","fam_hx","race"]:
    ct = pd.crosstab(train[v], train.y); chi, p, dof, _ = stats.chi2_contingency(ct)
    V = np.sqrt(chi/(ct.values.sum()*(min(ct.shape)-1)))
    out.append(dict(var=v, test=f"chi2({dof}) = {chi:.2f}", p=p, effect=f"V = {V:.2f}"))
biv = pd.DataFrame(out); biv.to_csv("outputs/bivariate_train.csv", index=False); biv

## 6. Regresi logistik multivariabel (Tabel 2)

In [ ]:
X = train[["age","male","bmi","waist","sbp","fam_hx","htn_hx","ever_smoke","pir","education","sedentary_h","chol_hx","dbp"]].copy()
for lab, r in [("race_NHB","NH Black"),("race_MA","Mexican American"),("race_OH","Other Hispanic"),("race_NHA","NH Asian"),("race_Oth","Other/Multi")]:
    X[lab] = (train.race == r).astype(int)                 # referensi: NH White
X["age"] /= 10; X["waist"] /= 10; X["sbp"] /= 10; X["dbp"] /= 10; X["bmi"] /= 5
dd = pd.concat([X, train.y], axis=1).dropna()           # analisis kasus lengkap
y = dd.y.values; Xm = np.column_stack([np.ones(len(dd)), dd.drop(columns="y").values])
b = np.zeros(Xm.shape[1])
for _ in range(100):                                    # Newton-Raphson (IRLS)
    p = 1/(1+np.exp(-Xm@b)); H = Xm.T@(Xm*(p*(1-p))[:, None]); step = np.linalg.solve(H, Xm.T@(y-p)); b += step
    if np.abs(step).max() < 1e-10: break
se = np.sqrt(np.diag(np.linalg.inv(H)))
ors = pd.DataFrame(dict(term=["const"]+list(X.columns), OR=np.exp(b), lo=np.exp(b-1.96*se), hi=np.exp(b+1.96*se),
                        p=2*stats.norm.sf(np.abs(b/se))))
print("n =", len(dd), "kasus =", int(y.sum()))
ors.to_csv("outputs/logit_or_train.csv", index=False); ors.round(3)

## 7. Model prediksi dan validasi temporal (Tabel 3)

In [ ]:
def pre(num, scale=True):
    numt = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
    return ColumnTransformer([("num", Pipeline(numt), num),
                              ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])

def models(num):
    return {
     "LR":  (Pipeline([("p", pre(num)), ("m", LogisticRegression(max_iter=3000))]), {"m__C": [0.01, 0.1, 1, 10]}),
     "RF":  (Pipeline([("p", pre(num, False)), ("m", RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=SEED))]),
             {"m__max_depth": [4, 6, 8], "m__min_samples_leaf": [20, 50]}),
     "GB":  (Pipeline([("p", pre(num, False)), ("m", HistGradientBoostingClassifier(random_state=SEED, early_stopping=False))]),
             {"m__learning_rate": [0.03, 0.1], "m__max_depth": [2, 3], "m__max_iter": [150, 300], "m__l2_regularization": [1.0]}),
     "MLP": (Pipeline([("p", pre(num)), ("m", MLPClassifier(max_iter=300, early_stopping=False, learning_rate_init=1e-3, random_state=SEED))]),
             {"m__hidden_layer_sizes": [(8,), (16,), (32, 16)], "m__alpha": [0.1, 1.0, 3.0]}),
    }

def cal_slope_int(y, p):
    p = np.clip(p, 1e-6, 1-1e-6); lp = np.log(p/(1-p))
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y).coef_[0][0]
    a = 0.0
    for _ in range(50):
        q = 1/(1+np.exp(-(a+lp))); a += np.sum(y-q)/np.sum(q*(1-q))
    return slope, a

def metrics(y, p, thr):
    y = np.asarray(y); idx = np.arange(len(y)); bs = []
    for _ in range(1000):
        bb = rng.choice(idx, len(idx), replace=True)
        if y[bb].sum() > 0: bs.append(roc_auc_score(y[bb], p[bb]))
    pred = p >= thr
    tp = np.sum(pred&(y==1)); fn = np.sum(~pred&(y==1)); tn = np.sum(~pred&(y==0)); fp = np.sum(pred&(y==0))
    s, c = cal_slope_int(y, p)
    return dict(AUC=roc_auc_score(y, p), AUC_lo=np.percentile(bs, 2.5), AUC_hi=np.percentile(bs, 97.5),
                AUPRC=average_precision_score(y, p), Brier=brier_score_loss(y, p), Sens=tp/(tp+fn), Spec=tn/(tn+fp),
                PPV=tp/(tp+fp), NPV=tn/(tn+fn), Cal_slope=s, CITL=c, mean_pred=p.mean(), obs_rate=y.mean())

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
res, preds, fitted = [], {}, {}
for setname, num in [("Full", NUM_FULL), ("Reduced", NUM_RED)]:
    X = train[num+CAT]; y = train.y.values
    for mname, (pipe, grid) in models(num).items():
        gs = GridSearchCV(pipe, grid, scoring="roc_auc", cv=skf, n_jobs=-1).fit(X, y)
        best = gs.best_estimator_
        oof = cross_val_predict(best, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
        fpr, tpr, th = roc_curve(y, oof); thr = th[np.argmax(tpr >= 0.80)]     # ambang sensitivitas >=80%
        fitted[(setname, mname)] = (best, thr)
        r = metrics(y, oof, thr); r.update(set=setname, model=mname, cohort="Train-CV 2013-2016", params=str(gs.best_params_)); res.append(r)
        tests = [("Temporal 2017-2018", testJ)] + ([("Temporal 2021-2023", testL)] if setname == "Reduced" else [])
        for tname, T in tests:
            p = best.predict_proba(T[num+CAT])[:, 1]; preds[(setname, mname, tname)] = (T.y.values, p)
            r = metrics(T.y.values, p, thr); r.update(set=setname, model=mname, cohort=tname, params=str(gs.best_params_)); res.append(r)
        print(setname, mname, gs.best_params_)
R = pd.DataFrame(res); R.to_csv("outputs/results_all.csv", index=False)
pickle.dump(preds, open("outputs/preds.pkl", "wb"))
R[["set","model","cohort","AUC","AUC_lo","AUC_hi","AUPRC","Brier","Cal_slope","CITL","Sens","Spec","PPV","NPV"]].round(3)

## 8. Perbandingan AUC berpasangan terhadap LR (bootstrap 2.000 ulangan)

In [ ]:
rb = np.random.default_rng(1); comp = []
for coh in ["Temporal 2017-2018", "Temporal 2021-2023"]:
    y, pl = preds[("Reduced", "LR", coh)]
    for m in ["RF", "GB", "MLP"]:
        _, pm = preds[("Reduced", m, coh)]; ds = []
        for _ in range(2000):
            bb = rb.integers(0, len(y), len(y)); ds.append(roc_auc_score(y[bb], pm[bb]) - roc_auc_score(y[bb], pl[bb]))
        comp.append(dict(cohort=coh, model=m, dAUC=roc_auc_score(y, pm)-roc_auc_score(y, pl), lo=np.percentile(ds, 2.5), hi=np.percentile(ds, 97.5)))
y, pf = preds[("Full","LR","Temporal 2017-2018")]; _, pr = preds[("Reduced","LR","Temporal 2017-2018")]; ds = []
for _ in range(2000):
    bb = rb.integers(0, len(y), len(y)); ds.append(roc_auc_score(y[bb], pf[bb]) - roc_auc_score(y[bb], pr[bb]))
comp.append(dict(cohort="Temporal 2017-2018", model="LR full - reduced", dAUC=roc_auc_score(y, pf)-roc_auc_score(y, pr), lo=np.percentile(ds, 2.5), hi=np.percentile(ds, 97.5)))
pd.DataFrame(comp).to_csv("outputs/paired_auc.csv", index=False); pd.DataFrame(comp).round(3)

## 9. Rekalibrasi logistik pada siklus 2021–2023

In [ ]:
recal = []
for mname in ["LR", "RF", "GB", "MLP"]:
    y, p = preds[("Reduced", mname, "Temporal 2021-2023")]
    idx = rng.permutation(len(y)); a, b_ = idx[:len(y)//2], idx[len(y)//2:]
    pc = np.clip(p, 1e-6, 1-1e-6); lp = np.log(pc/(1-pc))
    m = LogisticRegression(C=1e6).fit(lp[a].reshape(-1, 1), y[a]); pr = m.predict_proba(lp[b_].reshape(-1, 1))[:, 1]
    s0, c0 = cal_slope_int(y[b_], p[b_]); s1, c1 = cal_slope_int(y[b_], pr)
    recal.append(dict(model=mname, Brier_before=brier_score_loss(y[b_], p[b_]), Brier_after=brier_score_loss(y[b_], pr),
                      slope_before=s0, slope_after=s1, CITL_before=c0, CITL_after=c1))
pd.DataFrame(recal).to_csv("outputs/recalibration_L.csv", index=False); pd.DataFrame(recal).round(3)

## 10. Permutation importance, subkelompok, dan AUC berbobot

In [ ]:
imp = []
for mname in ["GB", "LR"]:
    best, _ = fitted[("Full", mname)]
    pi = permutation_importance(best, testJ[NUM_FULL+CAT], testJ.y.values, scoring="roc_auc", n_repeats=20, random_state=SEED, n_jobs=-1)
    imp += [dict(model=mname, feature=fe, dAUC=mu, sd=sd) for fe, mu, sd in zip(NUM_FULL+CAT, pi.importances_mean, pi.importances_std)]
imp = pd.DataFrame(imp); imp.to_csv("outputs/perm_importance.csv", index=False)

y, p = preds[("Full", "GB", "Temporal 2017-2018")]; T = testJ.assign(p=p); sub = []
T["agecat"] = pd.cut(T.age, [19, 44, 64, 120], labels=["20-44", "45-64", ">=65"])
for col in ["race", "male", "agecat"]:
    for lv, g in T.groupby(col):
        if g.y.sum() >= 10: sub.append(dict(var=col, level=lv, n=len(g), cases=int(g.y.sum()), AUC=roc_auc_score(g.y, g.p)))
pd.DataFrame(sub).to_csv("outputs/subgroup_auc.csv", index=False)

wa = []
for (s, m, c), (yy, pp) in preds.items():
    w = (testJ if "2017" in c else testL).w.values
    wa.append(dict(set=s, model=m, cohort=c, AUC=roc_auc_score(yy, pp), AUC_weighted=roc_auc_score(yy, pp, sample_weight=w)))
pd.DataFrame(wa).to_csv("outputs/weighted_auc.csv", index=False)
display(pd.DataFrame(sub).round(3)); pd.DataFrame(wa).round(3)

## 11. Gambar

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
C = {"LR":"#2a78d6","RF":"#eb6834","GB":"#1baf7a","MLP":"#eda100"}; LS = {"LR":"-","RF":"--","GB":"-.","MLP":":"}
NM = {"LR":"Regresi logistik","RF":"Random forest","GB":"Gradient boosting","MLP":"MLP"}
panels = [("Full","Temporal 2017-2018","(a) Validasi temporal 2017–2018 (fitur lengkap)"),
          ("Reduced","Temporal 2021-2023","(b) Validasi temporal 2021–2023 (tanpa riwayat keluarga)")]
fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
for a, (s, c, t) in zip(ax, panels):
    for m in C:
        y, p = preds[(s, m, c)]; fpr, tpr, _ = roc_curve(y, p)
        a.plot(fpr, tpr, color=C[m], ls=LS[m], lw=2, label=f"{NM[m]} (AUC {roc_auc_score(y, p):.3f})")
    a.plot([0, 1], [0, 1], color="#c3c2b7", lw=1); a.set(xlabel="1 − spesifisitas", ylabel="Sensitivitas", title=t)
    a.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig("outputs/Gambar2_ROC.png", dpi=300); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
for a, (s, c, t) in zip(ax, panels):
    mx = 0
    for m in C:
        y, p = preds[(s, m, c)]; q = pd.qcut(p, 10, labels=False, duplicates="drop")
        g = pd.DataFrame(dict(y=y, p=p, q=q)).groupby("q").mean(); mx = max(mx, g.max().max())
        a.plot(g.p, g.y, color=C[m], ls=LS[m], lw=2, marker="o", ms=4, label=NM[m])
    a.plot([0, mx*1.05], [0, mx*1.05], color="#c3c2b7", lw=1)
    a.set(xlabel="Probabilitas prediksi (rerata desil)", ylabel="Proporsi teramati", title=t); a.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig("outputs/Gambar3_Kalibrasi.png", dpi=300); plt.show()

pi = imp[imp.model == "LR"].sort_values("dAUC")
fig, a = plt.subplots(figsize=(6, 4.4))
a.barh(pi.feature, pi.dAUC, xerr=pi.sd, color="#2a78d6", height=0.6)
a.set(xlabel="Penurunan AUC saat fitur diacak (data 2017–2018)")
fig.tight_layout(); fig.savefig("outputs/Gambar4_Importance.png", dpi=300); plt.show()

## Catatan reproduksibilitas

- Python ≥3.10, scikit-learn ≥1.3, pandas, numpy, scipy, matplotlib (lihat `requirements.txt`).
- `random_state = 42` dipakai di seluruh model dan resampling; angka bootstrap dapat berbeda pada desimal ketiga antarversi pustaka.
- Gradient boosting memakai `HistGradientBoostingClassifier` dari scikit-learn.